In [1]:
from mgdraw_v08_detection_reader_npz import DetectionDataStorage
import toml
import subprocess
import numpy as np
import os

### Diverse

In [9]:
def orthogonalize_v(u, v):
    """
    Make v orthogonal to u (Gram–Schmidt), keeping u fixed.

    Parameters:
        u (array-like): reference vector
        v (array-like): vector to orthogonalize

    Returns:
        v_orth (np.ndarray): normalized vector orthogonal to u
    """
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)

    # Normalize u (important!)
    u_hat = u / np.linalg.norm(u)

    # Remove projection of v onto u
    v_orth = v - np.dot(v, u_hat) * u_hat

    # Normalize result
    v_orth /= np.linalg.norm(v_orth)

    return v_orth

def closest_point_on_plane(P, u, v):
    """
    Compute the point on the plane closest to the origin.

    Plane defined by: P + s*u + t*v

    Parameters:
        P (array-like): known point on plane
        u, v (array-like): spanning vectors

    Returns:
        P_new (np.ndarray): closest point to (0,0,0) on the plane
    """
    P = np.array(P, dtype=float)
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)

    # Build system: A [s, t]^T = b
    A = np.array([
        [np.dot(u, u), np.dot(u, v)],
        [np.dot(v, u), np.dot(v, v)]
    ])

    b = -np.array([
        np.dot(u, P),
        np.dot(v, P)
    ])

    # Solve for s, t
    s, t = np.linalg.solve(A, b)

    # Compute closest point
    P_new = P + s * u + t * v

    return P_new

In [23]:
script_dir = "/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/scintillator_interactions_neutrons_1e9"
new_path = "/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/scintillator_interactions_neutrons_1e9_npz"
files = os.listdir(script_dir)


for file in files:
    if "scintillator_interactions" in file:
        if True:
            
            NCASE = [] # 0
            ICODE = [] # 1
            JTRACK = [] #2
            KPART = [] # 3
            
            LLOUSE_list = [] # 4
            ICHTAR = [] # 5
            IBTAR = [] # 6
            Tki = [] # 7
            ETRACK_m_AM_TRACK = [] # 8
            XSCO = [] # 9
            YSCO = [] # 10
            ZSCO = [] # 11
            MREG = [] # 12
            LTRACK = [] # 13
            ATRACK = [] # 14
            XSCO_prod = [] # 15
            YSCO_prod = [] # 16
            ZSCO_prod = [] #17
            
            
            f = open(script_dir + "/" + file, "r")
            for line in f:
                elements = np.array(line.split())
                elements = elements.astype(float)

                #elements = elements[1:]
                
                if len(elements) == 18 - 3: #20
                    #print("Her")

                    NCASE.append(elements[0])
                    ICODE.append(elements[1])
                    JTRACK.append(elements[2])
                    KPART.append(elements[3])
                    LLOUSE_list.append(elements[4])
                    ICHTAR.append(elements[5])
                    IBTAR.append(elements[6])
                    Tki.append(elements[7]) #+2
                    ETRACK_m_AM_TRACK.append(elements[8])
                    XSCO.append(elements[9])
                    YSCO.append(elements[10])
                    ZSCO.append(elements[11])
                    MREG.append(elements[12])
                    LTRACK.append(elements[13])
                    ATRACK.append(elements[14])
                    XSCO_prod.append(0.0)#(elements[15])
                    YSCO_prod.append(0.0)#(elements[16])
                    ZSCO_prod.append(0.0)#(elements[17])
                
                
            # OR compressed version
            filename = file.replace("txt","npz")
            np.savez_compressed(new_path + "/" + filename,ncase=np.array(NCASE), icode=np.array(ICODE),jtrack=np.array(JTRACK),kpart=np.array(KPART),llouse_list=np.array(LLOUSE_list),ICHTAR=np.array(ICHTAR), IBTAR=np.array(IBTAR),tki=np.array(Tki),etrack_m_am_track=np.array(ETRACK_m_AM_TRACK),XSCO=np.array(XSCO),YSCO=np.array(YSCO),ZSCO=np.array(ZSCO),mreg=np.array(MREG),ltrack = np.array(LTRACK), atrack = np.array(ATRACK),XSCO_prod=np.array(XSCO_prod), YSCO_prod=np.array(YSCO_prod), ZSCO_prod=np.array(ZSCO_prod))
            
            
            # Tester datamengde
            #shutil.move(file, newpath + "/scintillator_interactions"+ "/" + file) 
            
            #os.remove(script_dir + "/"+ file) 
            

### Create only one SBP

In [25]:
scin_int_path = "/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/scintillator_interactions_neutrons_1e9_npz"
#"/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/scintillator_interactions_retry_npz"
primaries_per_spawn = 2944300#1e7
max_files = 10

file = DetectionDataStorage(scin_int_path, primaries_per_spawn=primaries_per_spawn, max_files=max_files)

file.load_data()
file.event_builder()
file.convert_to_Hunters_sbp_input()
#file.mgdraw_to_csv_builder()

Reading 10 scintillator_interactions.txt-files
18
18
18
18
18
18
18
18
18
18
10
Total time used on data collection: 15.78 s

-----------EVENT BUILDING-----------
Irregular gamma chains found: 616 / 17536
Irregular neutron chains found: 116264 / 574003
Event building complete, time used: 13.31 s
/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/scintillator_interactions_neutrons_1e9_npz_usrdef.out sucessfully created.


In [29]:
datfile_path = "/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/04_180.dat"
    
datfile = open(datfile_path, "r")
lines_datfile = datfile.readlines()[3]
elements = lines_datfile.split()



usrdef_file = "/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/scintillator_interactions_neutrons_1e9_npz_usrdef.out"
#"/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/scintillator_interactions_retry_npz_usrdef_1.out"
    
    
    ### OBS NEED to take into account more than just the depth 
xDir = float(elements[8])#0.832#float(elements[8])
yDir = float(elements[9])#-0.554#float(elements[9])
zDir = float(elements[10])#0.0 #float(elements[10])

P = [float(elements[1]), float(elements[2]), float(elements[3])]

    # Use only to find imaging plane origin
v_vec = [0.0, 0.0, 1.0]
u_vec = [xDir, yDir, 0.0]
normal_vec = np.cross(v_vec, u_vec)

origin = closest_point_on_plane(P, u = u_vec, v = v_vec) #[-3.0, 6.2, 2.0] #closest_point_on_plane(P, u = u_vec, v = v_vec)

toml_path = "/scratch/Anna/Proton_dose_reconstruction_vJune26/my_config.toml"
toml_file = toml.load(toml_path)
toml_file["plane"]["origin"] = origin
toml_file["plane"]["normal"] = normal_vec#[np.sin(beam_angle), -np.cos(beam_angle), -0.0] #normal_vec
toml_file["plane"]["u_axis"] = u_vec#[-np.cos(beam_angle), -np.sin(beam_angle), 0.0]#u_vec
toml_file["plane"]["v_axis"] = v_vec#[0.0,0.0,1.0]#v_vec
toml_file["io"]["input_path"] = usrdef_file
    
    
output_path =  "/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/"
toml_file["io"]["output_path"] = output_path + "output_1e9_neutrons.h5"

new_toml_path = output_path + "/my_config_1e9_neutrons.toml"
with open(new_toml_path, "w") as f:
        toml.dump(toml_file, f)


In [32]:
command = ["ng-run /mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/my_config_1e9_neutrons.toml"]
subprocess.run(command, shell=True)

[run] config = /mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/my_config_1e9_neutrons.toml
[run] neutrons=True gammas=True fast=False list=True
[run] input=/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/scintillator_interactions_neutrons_1e9_npz_usrdef.out -> output=/mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/output_1e9_neutrons.h5

[stage1] Raw events → hits
[stage1] Using staged path: raw events → hits → shaped → typed
[stage1] raw_events_total=119294 raw_events_after_filters=119273 raw_events_rejected_unreconstructable=21

[stage2] Hits → shaped events → typed events
[stage2] shaper: total_events_in=119273 shaped_n=118708 shaped_g=565
[stage2] Example shaped events (up to first 3):
    species=g n_hits=3 types=['g', 'g', 'g'] t_ns=[0.85, 0.852, 1.181] L=[1.217, 0.263, 0.056]
    species=g n_hits=3 types=['g', 'g', 'g'] t_ns=[1.69, 1.706, 2.016] L=[0.018, 0.016, 0.041]
    species=g n_hits=3 types=['g'

SBP:   0%|          | 0/186 [00:00<?, ?cone/s]

SBP: 46166/47729 cones intersected the plane
[stage4] Recon summed image stats (n): min= 0.0 max= 676.0 sum= 26090484.0 shape= (501, 501)
[stage4] Imaging gammas...
SBP: 163/186 cones intersected the plane
[stage4] Recon summed image stats (g): min= 0.0 max= 6.0 sum= 94092.0 shape= (501, 501)
[stage4] SUMMED Recon summed image stats (all): min= 0.0 max= 676.0 sum= 26184580.0 shape= (501, 501)


SBP: 100%|██████████| 186/186 [00:00<00:00, 2755.17cone/s]



[diagnostics] Counter summary:
  cones_checked_delta_theta = 47915
  cones_checked_delta_theta_g = 186
  cones_checked_delta_theta_n = 47729
  cones_checked_incident_energy = 47915
  cones_checked_incident_energy_g = 186
  cones_checked_incident_energy_n = 47729
  cones_kept = 47915
  cones_kept_g = 186
  cones_kept_n = 47729
  cones_n_recoil_carbon = 0
  cones_n_recoil_proton = 47729
  cones_n_recoil_unknown = 0
  events_after_filters = 54613
  events_cone_build_attempted = 54613
  events_cone_build_attempted_g = 186
  events_cone_build_attempted_n = 54427
  events_cone_build_failed = 6698
  events_cone_build_failed_n = 6698
  events_cone_build_success = 47915
  events_cone_build_success_g = 186
  events_cone_build_success_n = 47729
  events_rejected_L1_min_n = 30790
  events_rejected_L2_min_n = 32544
  events_rejected_L_any_min_g = 379
  events_rejected_filters = 64660
  events_rejected_tof_window = 947
  events_rejected_tof_window_n = 947
  events_total_for_filters = 119273
  event

CompletedProcess(args=['ng-run /mnt/medisinskfysikk/Anna/Study2_DoseReconstruction/Evaluate_two_step_sim/my_config_1e9_neutrons.toml'], returncode=0)